# 02. Imbalanced Classification Modeling

1번 노트북이 생성한 outputs/pkl/modeling_dataset.pkl을 읽어 양품/불량 분류 모델을 비교합니다.

평가 지표:
- F1
- PR-AUC / Average Precision
- Precision, Recall
- ROC-AUC
- Balanced Accuracy 
불량 비율이 약 3%이므로 PR-AUC와 Recall/F1을 Accuracy보다 우선합니다.

In [ ]:
import sys
print(sys.executable)
print(sys.version)

# 필요 패키지 예시:
# .venv/bin/pip install pandas numpy matplotlib seaborn scikit-learn joblib
# 선택 모델:
# .venv/bin/pip install imbalanced-learn xgboost lightgbm catboost pytorch-tabnet torch

In [ ]:
from pathlib import Path
import importlib.util
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import OneClassSVM
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, confusion_matrix, classification_report, precision_recall_curve
)

sns.set_theme(style='whitegrid', font='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd()
PKL_PATH = ROOT / 'outputs' / 'pkl' / 'modeling_dataset.pkl'
MODEL_OUT_DIR = ROOT / 'outputs' / 'modeling'
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
RUN_HEAVY_MODELS = False

## 1. pkl 로드 및 split

In [ ]:
dataset = pd.read_pickle(PKL_PATH)
X_tree = dataset['X_tree']
X_lr = dataset['X_lr_vif']
y = dataset['y'].astype(int)

print('X_tree:', X_tree.shape)
print('X_lr:', X_lr.shape)
print('class distribution:')
print(y.value_counts(normalize=False).sort_index())
print(y.value_counts(normalize=True).sort_index())

idx_train_valid, idx_test = train_test_split(y.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
idx_train, idx_valid = train_test_split(idx_train_valid, test_size=0.25, stratify=y.loc[idx_train_valid], random_state=RANDOM_STATE)
print(len(idx_train), len(idx_valid), len(idx_test))
print('train defect rate:', y.loc[idx_train].mean())
print('valid defect rate:', y.loc[idx_valid].mean())
print('test defect rate:', y.loc[idx_test].mean())

## 2. 평가 함수

In [ ]:
def find_best_threshold(y_true, score):
    precision, recall, thresholds = precision_recall_curve(y_true, score)
    if len(thresholds) == 0:
        return 0.5
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return float(thresholds[best_idx])

def get_scores(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        s = model.decision_function(X)
        return (s - np.nanmin(s)) / max(np.nanmax(s) - np.nanmin(s), 1e-12)
    pred = model.predict(X)
    return pred.astype(float)

def metric_row(name, y_true, score, threshold):
    pred = (score >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    row = {
        'model': name,
        'threshold': threshold,
        'f1': f1_score(y_true, pred, zero_division=0),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'pr_auc': average_precision_score(y_true, score),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'tn': cm[0, 0], 'fp': cm[0, 1], 'fn': cm[1, 0], 'tp': cm[1, 1],
    }
    try:
        row['roc_auc'] = roc_auc_score(y_true, score)
    except Exception:
        row['roc_auc'] = np.nan
    return row

def evaluate_supervised(name, model, X, idx_train, idx_valid, idx_test):
    X_tr, y_tr = X.loc[idx_train], y.loc[idx_train]
    X_va, y_va = X.loc[idx_valid], y.loc[idx_valid]
    X_te, y_te = X.loc[idx_test], y.loc[idx_test]
    fitted = clone(model)
    fitted.fit(X_tr, y_tr)
    valid_score = get_scores(fitted, X_va)
    threshold = find_best_threshold(y_va, valid_score)
    test_score = get_scores(fitted, X_te)
    row = metric_row(name, y_te, test_score, threshold)
    return row, fitted, test_score

def evaluate_oneclass(name, model, X, idx_train, idx_valid, idx_test):
    y_tr = y.loc[idx_train]
    normal_idx = y_tr[y_tr == 0].index
    fitted = clone(model)
    fitted.fit(X.loc[normal_idx])
    valid_score = -fitted.decision_function(X.loc[idx_valid])
    test_score = -fitted.decision_function(X.loc[idx_test])
    threshold = find_best_threshold(y.loc[idx_valid], valid_score)
    row = metric_row(name, y.loc[idx_test], test_score, threshold)
    return row, fitted, test_score

## 3. 모델 registry

설치되지 않은 외부 패키지는 자동으로 skip합니다.

In [ ]:
def has_pkg(pkg):
    return importlib.util.find_spec(pkg) is not None

models = []
models.append(('Dummy_stratified', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE), 'tree', 'supervised'))
models.append(('LogisticRegression_balanced', Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=3000, solver='lbfgs', random_state=RANDOM_STATE))
]), 'lr', 'supervised'))
models.extend([
    ('DecisionTree_balanced', DecisionTreeClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE), 'tree', 'supervised'),
    ('RandomForest_balanced', RandomForestClassifier(n_estimators=400, class_weight='balanced_subsample', min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE), 'tree', 'supervised'),
    ('ExtraTrees_balanced', ExtraTreesClassifier(n_estimators=400, class_weight='balanced', min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE), 'tree', 'supervised'),
    ('GradientBoosting', GradientBoostingClassifier(random_state=RANDOM_STATE), 'tree', 'supervised'),
    ('HistGradientBoosting', HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=RANDOM_STATE), 'tree', 'supervised'),
])
models.append(('OneClassSVM_normal_only', Pipeline([
    ('scaler', StandardScaler()),
    ('clf', OneClassSVM(kernel='rbf', nu=0.035, gamma='scale'))
]), 'lr', 'oneclass'))

if has_pkg('imblearn'):
    from imblearn.ensemble import BalancedRandomForestClassifier, RUSBoostClassifier, EasyEnsembleClassifier
    models.extend([
        ('BalancedRandomForest', BalancedRandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1), 'tree', 'supervised'),
        ('RUSBoost', RUSBoostClassifier(n_estimators=300, learning_rate=0.05, random_state=RANDOM_STATE), 'tree', 'supervised'),
        ('EasyEnsemble', EasyEnsembleClassifier(n_estimators=20, random_state=RANDOM_STATE, n_jobs=-1), 'tree', 'supervised'),
    ])
else:
    print('skip imbalanced-learn models: imblearn not installed')

if has_pkg('xgboost'):
    from xgboost import XGBClassifier
    scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
    models.append(('XGBoost_weighted', XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
        objective='binary:logistic', eval_metric='aucpr', scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, n_jobs=-1
    ), 'tree', 'supervised'))
else:
    print('skip XGBoost: xgboost not installed')

if has_pkg('lightgbm'):
    from lightgbm import LGBMClassifier
    models.append(('LightGBM_balanced', LGBMClassifier(
        n_estimators=600, learning_rate=0.03, num_leaves=31, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
    ), 'tree', 'supervised'))
else:
    print('skip LightGBM: lightgbm not installed')

if has_pkg('catboost'):
    from catboost import CatBoostClassifier
    models.append(('CatBoost_balanced', CatBoostClassifier(
        iterations=600, learning_rate=0.03, depth=5, loss_function='Logloss', eval_metric='PRAUC',
        auto_class_weights='Balanced', random_seed=RANDOM_STATE, verbose=False
    ), 'tree', 'supervised'))
else:
    print('skip CatBoost: catboost not installed')

if RUN_HEAVY_MODELS and has_pkg('pytorch_tabnet'):
    from pytorch_tabnet.tab_model import TabNetClassifier
    models.append(('TabNet', TabNetClassifier(seed=RANDOM_STATE, verbose=0), 'tree', 'supervised'))
elif not RUN_HEAVY_MODELS:
    print('skip TabNet: RUN_HEAVY_MODELS=False')

print('models to run:', [m[0] for m in models])

## 4. 모델 학습 및 성능 비교

In [ ]:
results = []
fitted_models = {}
test_scores = {}

for name, model, data_key, mode in models:
    X = X_lr if data_key == 'lr' else X_tree
    print(f'\n=== {name} ===')
    try:
        if mode == 'oneclass':
            row, fitted, score = evaluate_oneclass(name, model, X, idx_train, idx_valid, idx_test)
        else:
            row, fitted, score = evaluate_supervised(name, model, X, idx_train, idx_valid, idx_test)
        results.append(row)
        fitted_models[name] = fitted
        test_scores[name] = score
        print({k: row[k] for k in ['f1','precision','recall','pr_auc','threshold']})
    except Exception as e:
        print('FAILED:', repr(e))
        results.append({'model': name, 'status': f'failed: {e}'})

results_df = pd.DataFrame(results)
results_sorted = results_df.sort_values(['pr_auc', 'f1'], ascending=False, na_position='last')
display(results_sorted)
results_sorted.to_csv(MODEL_OUT_DIR / 'model_performance_comparison.csv', index=False, encoding='utf-8-sig')
print('saved:', MODEL_OUT_DIR / 'model_performance_comparison.csv')

In [ ]:
plot_df = results_sorted.dropna(subset=['f1', 'pr_auc']).copy()
if len(plot_df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.barplot(data=plot_df, y='model', x='pr_auc', ax=axes[0], color='#4C78A8')
    axes[0].set_title('PR-AUC 비교')
    sns.barplot(data=plot_df, y='model', x='f1', ax=axes[1], color='#F58518')
    axes[1].set_title('F1 비교')
    fig.tight_layout()
    fig.savefig(MODEL_OUT_DIR / 'model_metric_comparison.png', dpi=150)
    plt.show()

## 5. Best model 상세 확인

In [ ]:
valid_results = results_sorted.dropna(subset=['pr_auc', 'f1'])
if len(valid_results):
    best_name = valid_results.iloc[0]['model']
    best_threshold = float(valid_results.iloc[0]['threshold'])
    best_score = test_scores[best_name]
    y_test = y.loc[idx_test]
    y_pred = (best_score >= best_threshold).astype(int)

    print('Best model:', best_name)
    print('Threshold:', best_threshold)
    print(classification_report(y_test, y_pred, target_names=['양품', '불량'], zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['양품 예측','불량 예측'], yticklabels=['실제 양품','실제 불량'], ax=ax)
    ax.set_title(f'Confusion Matrix: {best_name}')
    fig.tight_layout()
    fig.savefig(MODEL_OUT_DIR / 'best_model_confusion_matrix.png', dpi=150)
    plt.show()

    precision, recall, thresholds = precision_recall_curve(y_test, best_score)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall, precision)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'Precision-Recall Curve: {best_name}')
    fig.tight_layout()
    fig.savefig(MODEL_OUT_DIR / 'best_model_pr_curve.png', dpi=150)
    plt.show()

## 6. Feature importance / coefficient 확인

In [ ]:
if len(valid_results):
    model = fitted_models[best_name]
    X_used = X_lr if 'LogisticRegression' in best_name or 'OneClassSVM' in best_name else X_tree
    feature_names = X_used.columns

    importance = None
    estimator = model
    if hasattr(model, 'named_steps'):
        estimator = model.named_steps.get('clf', model)

    if hasattr(estimator, 'feature_importances_'):
        importance = pd.DataFrame({'feature': feature_names, 'importance': estimator.feature_importances_})
    elif hasattr(estimator, 'coef_'):
        importance = pd.DataFrame({'feature': feature_names, 'importance': np.ravel(estimator.coef_)})

    if importance is not None:
        importance['abs_importance'] = importance['importance'].abs()
        importance = importance.sort_values('abs_importance', ascending=False)
        display(importance.head(30))
        importance.to_csv(MODEL_OUT_DIR / f'{best_name}_importance.csv', index=False, encoding='utf-8-sig')

        fig, ax = plt.subplots(figsize=(10, 8))
        top_imp = importance.head(25).sort_values('abs_importance')
        sns.barplot(data=top_imp, x='abs_importance', y='feature', ax=ax, color='#72B7B2')
        ax.set_title(f'Top Feature Importance: {best_name}')
        fig.tight_layout()
        fig.savefig(MODEL_OUT_DIR / 'best_model_feature_importance.png', dpi=150)
        plt.show()
    else:
        print('선택된 best model은 importance/coefficient를 직접 제공하지 않습니다.')

## 7. 해석 메모

- 불량이 3% 수준이므로 Accuracy는 결과표에서 제외했습니다.
- 실제 운영 목적이 “불량 누락 최소화”라면 threshold를 더 낮춰 Recall을 높이고, Precision 하락을 감수하는 정책을 검토하세요.
- 최종 제출 전에는 test set을 여러 번 보며 threshold를 조정하지 않는 것이 좋습니다. threshold는 valid set에서 결정하고 test set은 최종 확인 용도로 유지합니다.